In [0]:
import requests
from bs4 import BeautifulSoup
import os
from urllib.parse import urljoin

BASE_URL = "https://download.bls.gov/pub/time.series/pr/"
HEADERS = {"User-Agent": "Prathamesh Palaskar contact: pratham.palaskar@gmail.com"}
LANDING_DIR = "/Volumes/rearc_bls/bronze/raw_landing/pr"

os.makedirs(LANDING_DIR, exist_ok=True)

resp = requests.get(BASE_URL, headers=HEADERS)
resp.raise_for_status()
soup = BeautifulSoup(resp.text, "html.parser")

hrefs = [a["href"] for a in soup.find_all("a") if not a["href"].endswith("/")]

for href in hrefs:
    file_url = urljoin(BASE_URL, href)          # <-- handles both absolute and relative hrefs correctly
    fname = os.path.basename(file_url)          # <-- derive a clean local filename regardless of href format
    local_path = os.path.join(LANDING_DIR, fname)

    head = requests.head(file_url, headers=HEADERS)
    remote_size = int(head.headers.get("Content-Length", -1))

    if os.path.exists(local_path) and os.path.getsize(local_path) == remote_size:
        print(f"Skipping unchanged: {fname}")
        continue

    r = requests.get(file_url, headers=HEADERS)
    r.raise_for_status()
    with open(local_path, "wb") as f:
        f.write(r.content)
    print(f"Downloaded/updated {fname}")